In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [2]:
%load_ext autoreload
%autoreload 2

import re
import os
import tempfile
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
import python.MS_ML_prioritization as mlp
from mltrail import Registry

import python.functions as fn
from get_library import get_df   # CDD Vault collection export
from python.MS_build_ML import PARAMS, DATA, OUTPUT  

# from tdc.multi_pred import DTI

## 0. Imports

In [3]:
# -------------------
# CONFIG Params
# -------------------

import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = params = SimpleNamespace(**_cfg)

globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')

# -------------------
# Dropbox access — DROPBOX_BACKEND picks how the Dropbox paths below are read: 'rclone' pulls
# straight from Dropbox (no laptop, no tunnel); 'ssh' streams the laptop's mirror over the
# reverse tunnel (must be up). See wiki "Dropbox access".
# -------------------

os.environ.setdefault('DROPBOX_SSH_HOST', DROPBOX_SSH_HOST)
os.environ.setdefault('DROPBOX_SSH_PORT', str(DROPBOX_SSH_PORT))
os.environ['DROPBOX_REMOTE']     = DROPBOX_REMOTE       # assign, not setdefault, so config always wins
os.environ['DROPBOX_LOCAL_ROOT'] = DROPBOX_LOCAL_ROOT
_use_rclone = DROPBOX_BACKEND == 'rclone'
_use_ssh  = True                                        # Dropbox is remote -> wrap its paths with _dbx
_dbx      = fn.open_rclone if _use_rclone else fn.open_dropbox
_fbx_open = _dbx
_fbx_glob = fn.glob_rclone if _use_rclone else fn.glob_dropbox
print(f'> Dropbox backend: {DROPBOX_BACKEND}')

# -------------------
# MLTrail Params
# -------------------

registry = Registry.from_default()   # shared vault from MLTrail config; reuse for every add/predict


> loaded 82 params from config/config.yaml
> Dropbox backend: rclone


In [4]:
%%time
## data — library + proteomics + chemistry features (see python/MS_build_ML.py:DATA)
data = DATA()
data.load_chemical_lib_df(params)                                  # serac_df + validated / devalidated targets
## cm2rm now comes from config CM2RM_PARTS — any subset of contaminants (the CONTAMINANTS csv) |
## control_compounds (CONTROLS) | fbx_independent (Px_Ligase_dependent == 0 or Px_validated_WT == 0)
data.get_contaminants_and_controls(params)                         # parts=(...) here would override the config
data.load_proteomics_data(params)                                  # DFRAW_OVERWRITE -> rebuild from the tranches, else read the parquets
data.compute_features(params)                                      # FEATURES_TYPE -> MF_features (+ MF morgan bits)

## OpenTargets disease scores + the big-pharma-relevant subset (see python/MS_build_ML.py:DATA)
data.load_opentargets(params)   # ot_df, cached to OT_CACHE — delete that file to force a recompute from OT_ROOT
data.filter_pharma_targets()    # ot_pharma, via the module-level PHARMA_PRIORITY_AREAS / PHARMA_DROP_AREAS

## transition shim — bare names used by the cells below

## transition shim — bare names used by the cells below
serac_df, MS, df_raw, MF, MF_features = data.serac_df, data.MS, data.df_raw, data.MF, data.MF_features
control_compounds, contaminants, cm2rm = data.control_compounds, data.contaminants, data.cm2rm
fbx_independent = data.fbx_independent
validated_targets, devalidated_targets = data.validated_targets, data.devalidated_targets
ot_df, ot_pharma = data.ot_df, data.ot_pharma
# ot_df.to_csv('data/patent/20260609_OpenTarget_list.csv', sep=',', index=False)

> Chemical lib dim: (14168, 13) | 40 validated / 2772 devalidated targets
> 2 control + 81 contaminant compounds (from config) + 1026 fbx independent | cm2rm = contaminants = 81
> loaded cached MS (7222, 5) | df_raw (66186971, 12)
> DROP_CM2RM: dropped 38 compounds (381,674 df_raw rows) / 0 MS compounds
> MS derived from df_raw: 7,222 compounds | 153 change activity bin | 2 keep the delivered value (no df_raw row)
> checking smiles


100%|██████████| 14168/14168 [00:03<00:00, 3888.24it/s]


> FEATURES_TYPE: H237


H237 descriptors: 100%|██████████| 128/128 [00:08<00:00, 15.54it/s]


> 0 smiles failed | MF (14168, 2049) | MF_features (14086, 4470)
> loaded cached output/MS/opentargets_target_disease.parquet
> 232,004 (target, disease) rows / 11,584 targets resolved
> 78,935 / 232,004 (target, disease) rows pass the pharma filter | 9,105 unique targets
  17,071  cancer or benign tumor
   3,890  cardiovascular disease
   2,504  disorder of visual system
   3,466  endocrine system disease
   3,416  hematologic disease
   3,938  immune system disease
   3,710  infectious disease
   8,708  musculoskeletal or connective tissue disease
  25,393  nervous system disease
   9,849  nutritional or metabolic disease
   1,704  psychiatric disorder
   2,357  respiratory or thoracic disease
CPU times: user 1min 24s, sys: 34.9 s, total: 1min 59s
Wall time: 1min 44s


## 1. Prediction and Prioritization

#### 20260528 - Prioritizing virtual & library compounds vs PCSK9
We will use the 20260513 trained models to proritize compounds coming from a virtual enumeration and unscreen compounds from our library

In [ ]:
## get enamine virtual enum - convert sdf to smiles and extract relevant fields/columns
enum_df = rdkit_tools.get_smiles_df_from_enum(ENAMINE_20260513)
enum_df['compound']  = enum_df['R1_Code'] + '_' + enum_df['R2_Code']
enum_df['filename'] = Path(ENAMINE_20260513).stem
enum_df['smiles'] = enum_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum_df.head(1)

In [ ]:
## get unscreened library:
lib = pd.read_csv(PX_SCREEN_LIB).rename(columns={'Molecule Name':'compound','SMILES':'smiles','Lib ID':'lib_id','Px_screened_source':'source'})
lib['smiles'] = lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
print(lib['Px_screened_anywhere'].unique())
lib['Px_sreened'] = 0
lib.loc[lib['Px_screened_anywhere']=='yes','Px_sreened'] = 1
lib = lib[['compound','smiles','lib_id','source','Px_sreened']]

# select uncreened compounds from enamine 4 lib id
# lib = lib[ (lib['lib_id']=='Enamine 4') & (lib['Px_sreened']==0) ]
# print(lib.shape)
# lib.head(1)

# get all the screened compounds:
Px_screened_smiles = list(lib[lib['Px_sreened']==1]['smiles'])
len(Px_screened_smiles)

In [ ]:
## remove anything that could have been screened from the enumeration:
print(f"before {enum_df.shape}")
enum_df = enum_df[~enum_df['smiles'].isin(Px_screened_smiles)]
print(f"after {enum_df.shape}")

In [ ]:
## prioritize compounds:
## Implementation now lives in Scripts/ML_Reg.py (ML_Reg.MLReg_prioritize_compounds).
## The function loads a joblib bundle, computes H236 features on `ori_data`, and
## returns the top-N most-active compounds. See its docstring for details.
model_path = 'output/ML/trained_models/20260513/PCSK9_RF_H236.joblib'

pred_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum_df,
    model=model_path,
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
)


In [ ]:
test_outpath = DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf'
pred_df

## Sanity check: SMILES intersection between the written SDF and pred_df
from rdkit import Chem

# 1) Read back what we wrote
out_df = rdkit_tools.get_smiles_df_from_enum(test_outpath)

# 2) Re-canonicalize both sides defensively — different code paths
#    (MolToSmiles in get_smiles_df_from_enum vs convert_smiles_to_canonical
#    upstream of pred_df) usually agree, but a one-pass re-canonicalisation
#    eliminates any kekulization / stereo-perception drift.
def _canon(smi):
    if not isinstance(smi, str) or not smi:
        return None
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else None

out_smis  = set(out_df['smiles'].map(_canon).dropna())
pred_smis = set(pred_df['smiles'].map(_canon).dropna())

inter      = out_smis & pred_smis
only_in_out  = out_smis - pred_smis
only_in_pred = pred_smis - out_smis

print(f'SDF written         : {len(out_df):,} rows  ({len(out_smis):,} unique canonical SMILES)')
print(f'pred_df             : {len(pred_df):,} rows ({len(pred_smis):,} unique canonical SMILES)')
print(f'∩ (intersection)    : {len(inter):,}')
print(f'in SDF, NOT in pred : {len(only_in_out):,}')
print(f'in pred, NOT in SDF : {len(only_in_pred):,}')




In [ ]:
## write sdf in dropbox folder:
if pred_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred_df,
        source_dir=DROPBOX_ML+'virtual libraries', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_enum_NAr-pyrimidine_Enamine_4_pred.sdf',
        pred_col='predicted_label',
)

#### Prioritize 150 compounds from 114 virtual enumeration

In [ ]:
## get 114 virtual enumerations:
enumpath = 'data/enumeration/20260427/'
enum_fs = glob(enumpath+'*')

# get a unified file for prediction:
# combined_sdf, n_parts = rdkit_tools.combine_sdfs(enumpath, '20260518_114K_enum.sdf')
enum114_df = rdkit_tools.get_smiles_df_from_enum_dir(enumpath, v=True)
print(f'> parsed {enum114_df["filename"].nunique()} SDFs, {len(enum114_df):,} rows total')


# get smiles from this:
enum114_df['compound']  = enum114_df['R1_Code'] + '_' + enum114_df['R2_Code']
enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
enum114_df = enum114_df.drop_duplicates('smiles').reset_index(drop=True)

## remove compounds already present in other sets:
cm2rm = list(enum_df['smiles']) + Px_screened_smiles
print(f"before {enum114_df.shape}")
enum114_df = enum114_df[~enum114_df['smiles'].isin(cm2rm)]
print(f"before {enum114_df.shape}")


In [ ]:
## prioritize compounds:
pred114_df = ML_Reg.MLReg_prioritize_compounds(
    ori_data=enum114_df,
    model='output/ML/trained_models/20260513/PCSK9_RF_H236.joblib',
    top=300,
    features_n=None,        # use the bundle's saved featurizer name
    verbose=True,
)

In [ ]:
## write sdf in dropbox folder:
if pred114_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred114_df,
        source_dir='data/enumeration/20260427/', # exact path to sdf file which let's copy verbatim the stereochemistry and attributes
        out_path=DROPBOX_ML+'predictions/20260518_Enamine_114k_pred.sdf',
        pred_col='predicted_label',
)

#### 20260608 Re-Prioritize these compounds based on curated data
We remove all compounds that are found to be FBX indepent downmodulation as well as not validation in n=2

In [ ]:
## get 114 virtual enumerations:
enum750k_df = pd.read_csv('data/enumeration/20260608/Serac scaffolds broad enumeration.csv', sep='\t',).rename(columns={'SMILES':'smiles'})
enum750k_df['compound'] = 'X'+enum750k_df.index.astype(str)
enum750k_df.head(1)

In [ ]:
%%time 
## get 114 virtual enumerations:
enum750k_df = pd.read_csv('data/enumeration/20260608_PCSK9/Serac scaffolds broad enumeration.csv', sep='\t',).rename(columns={'SMILES':'smiles'})
enum750k_df['compound'] = 'X'+enum750k_df.index.astype(str)

# # enum114_df['smiles'] = enum114_df['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))

# ## prioritize compounds:
# pred114_df = ML_Reg.MLReg_prioritize_compounds(
#     ori_data=enum750k_df,
#     # model='output/ML/trained_models/20260513/PCSK9_RF_H236.joblib',
#     model='output/ML/trained_models/20260608_PCSK9/PCSK9_RF_H236.joblib',
#     top=1000,
#     features_n=None,        # use the bundle's saved featurizer name
#     verbose=True,
# )

In [ ]:
pred114_df.head(1)

In [ ]:
## write sdf in dropbox folder (SMILES-only, no template):
## tag_cols = the fields to embed as SDF tags (one each). 'compound' becomes the
## molecule title automatically. pred_col is just the single-tag shortcut for
## TEMPLATE mode; in SMILES-only mode use tag_cols for one-or-many fields.
if pred114_df.shape[0] > 0:
    n = rdkit_tools.write_filtered_enum_sdf(
        pred114_df,
        out_path=DROPBOX_ML+'20260608_Echoes_PCSK9_750k_enamine.sdf',
        tag_cols=['RSN', 'predicted_label'],
    )
    print(f'wrote {n} records')


#### 20260622 Prioritize compounds to hit ['PCSK9','TGFB1','EPHB4','BRD2','ANGPTL4'] and that are Single / Low

##### Params

In [ ]:
## params:
# out
enum     = 'data/enumeration/20260608_5Targets_SingleLow/Serac scaffolds broad enumeration_filtered.csv' # input
outMFsmi = 'output/enumeration/20260608_5Targets_SingleLow/MF_features_H236_full.parquet' # MF_features
out550k  = 'output/enumeration/20260608_5Targets_SingleLow/pred_logfc_select_genes.parquet' # the predicted logfc per target
out_sl   = 'output/enumeration/20260608_5Targets_SingleLow/pred_single_low.parquet'
# ml models
SELECT_GENES_DIR = 'output/ML/trained_models/20260617_select_genes' # the ML trained model per gene
SINGLELOW_MODEL  = 'output/ML/trained_models/20260623_SingleLow_Class/SingleLow_RF.joblib' # the ML model to predict single/low

In [ ]:
## get 114 virtual enumerations:
enum550k_df = pd.read_csv(enum, sep=',',).rename(columns={'mol_smi':'smiles'})
enum550k_df['compound'] = 'X'+enum550k_df.index.astype(str)
print(enum550k_df.shape)
enum550k_df.head(1)

##### Prediction

In [ ]:
## compute the FULL H236 feature universe + cache it (one expensive pass over the library).
## Do NOT prevalence-drop here: each trained model carries its own prevalence-filtered
## `feature_cols` (cut on its TRAINING set), and the prediction cell selects them per-model.
## A 550K-derived prevalence cut does NOT match the models' columns (it drops ~91 Morgan bits
## the models need -> KeyError). compute_H236_features already returns the full universe
## (F0..F2047 + physchem + MACCS + AP_0..AP2047), so feature selection is a downstream step.
if not os.path.exists(outMFsmi):
    MF_features = rdkit_tools.compute_H236_features(enum550k_df, v=True)
    MF_features = enum550k_df[['compound', 'smiles']].merge(MF_features, on='compound').drop_duplicates('compound')
    MF_features.to_parquet(outMFsmi, index=False)
else:
    MF_features = pd.read_parquet(outMFsmi)
print(MF_features.shape)

In [ ]:
## predict logfc/probabilities with each model
## RandomForest *regressors* (one per gene) -> each predicts a continuous logfc.
## Each bundle carries its own prevalence-filtered `feature_cols` (a subset of the full
## H236 universe in MF_features); select per-model, then predict. Features are computed
## once (cell above) and reused, so no 5x re-featurisation of the library.

pred550k_df = MF_features[['compound', 'smiles']].copy()
_Xkey = _X = None
for mp in sorted(glob(os.path.join(SELECT_GENES_DIR, '*.joblib'))):
    b = joblib.load(mp)
    gene, fcols = b['gene'], b['feature_cols']
    missing = [c for c in fcols if c not in MF_features.columns]
    assert not missing, f"{gene}: {len(missing)} feature_cols missing from MF_features (regenerate the full-universe cache); e.g. {missing[:5]}"
    if tuple(fcols) != _Xkey:                      # rebuild only when feature_cols change; keep
        _Xkey, _X = tuple(fcols), MF_features[fcols]   # column NAMES so sklearn validates alignment
    pred550k_df[gene] = b['model'].predict(_X)
    print(f"> {gene:8s} ({b.get('features')}, {len(fcols)} cols, n_train={b.get('n_train')}): "
          f"pred logfc [{pred550k_df[gene].min():.2f}, {pred550k_df[gene].max():.2f}] mean {pred550k_df[gene].mean():.2f}")

gene_cols = [c for c in pred550k_df.columns if c not in ('compound', 'smiles')]

pred550k_df.to_parquet(out550k, index=False)
print(f"\n> saved {len(pred550k_df):,} compounds x {len(gene_cols)} gene logfc predictions -> {out550k}")
pred550k_df[['compound'] + gene_cols].head()

In [ ]:
## predict + prioritize single/low (classifier) on the 550K — same full-H236 deploy pattern
## Loads the SingleLow bundle, selects its feature_cols from the full H236 universe (MF_features),
## ranks compounds by P(single/low). NOTE: classifier -> predict_proba[:, 1], not predict.

_b = joblib.load(SINGLELOW_MODEL)
_fcols = _b['feature_cols']
_missing = [c for c in _fcols if c not in MF_features.columns]
assert not _missing, f"{len(_missing)} feature_cols missing from MF_features (regenerate the full-universe cache); e.g. {_missing[:5]}"

pred_singlelow_df = MF_features[['compound', 'smiles']].copy()
pred_singlelow_df['p_single_low'] = _b['model'].predict_proba(MF_features[_fcols])[:, 1]   # named df -> sklearn validates cols
pred_singlelow_df = pred_singlelow_df.sort_values('p_single_low', ascending=False).reset_index(drop=True)

pred_singlelow_df.to_parquet(out_sl, index=False)
print(f"> {_b['task']} ({_b.get('features')}, {len(_fcols)} cols, n_train={_b.get('n_train')}): "
      f"P(single/low) range [{pred_singlelow_df['p_single_low'].min():.2f}, {pred_singlelow_df['p_single_low'].max():.2f}] "
      f"mean {pred_singlelow_df['p_single_low'].mean():.2f}")
print(f"> saved {len(pred_singlelow_df):,} ranked compounds -> {out_sl}")
pred_singlelow_df[['compound', 'p_single_low']].head()

##### Prioritization using all targets/labels

In [ ]:
pred550k_df = pd.read_parquet(out550k)
pred_singlelow_df = pd.read_parquet(out_sl)

final_pred = pd.merge(pred550k_df,pred_singlelow_df[['compound','smiles', 'p_single_low']])
final_pred = pd.merge(enum550k_df[['smiles','RSN']],final_pred)
print(final_pred.shape)
final_pred.head(1)

In [ ]:
## get Pareto optimal points
prio_all = stats_tools.top_k_pareto(
    final_pred, objectives=['ANGPTL4','BRD2','EPHB4','PCSK9','TGFB1','p_single_low'],
    k=1000, directions={'p_single_low': 'max'})   # the 5 logfc default to 'min'
prio_all.head(2)

In [ ]:
## Pairwise scatter matrix: selected top-1000 (azure) over full library (paper).
_objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
mlp.plot_pairwise_matrix(_objs, final_pred, prio_all, colors=SERAC_C);

In [ ]:
## Visualize directionality: per-objective KDE overlay (full library vs Pareto selections).
_objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
mlp.plot_objective_kde(
    _objs,
    [final_pred, prio_all, prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', 'with TGFB1', 'without TGFB1'],
);

In [ ]:
## Covariance analysis: Spearman correlation of the predicted objectives.
_objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
_fig, _corr = mlp.plot_objective_correlation(_objs, final_pred)

##### Prioritization removing TGFB1

In [ ]:
## get Pareto optimal points
_objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'p_single_low']
prio_noTGF = stats_tools.top_k_pareto(
    final_pred[final_pred['p_single_low']>0.63], objectives=_objs, # 
    k=800, directions={'p_single_low': 'max'})   # the 5 logfc default to 'min'
prio_noTGF.head(2)

In [ ]:
# ## Pairwise scatter matrix: selected top-1000 (azure) over full library (paper).
# mlp.plot_pairwise_matrix(_objs, final_pred, prio_noTGF, colors=SERAC_C);

In [ ]:
## plot logfc distribution of actual proteomics (df_raw): predicted (550K) vs actual training labels.
## 'actual' uses the exact training transform (per-gene 1-99% winsorize -> compound-level mean).
_g5 = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1']
_raw = pads.dataset(
    'autoresearch/optimize_SingleLow_Classification/20260622_df_raw.parquet', format='parquet'
).to_table(columns=['compound', 'genes', 'logfc'], filter=pads.field('genes').isin(_g5)).to_pandas()

_actual = {}
for g in _g5:
    sub = _raw[_raw['genes'] == g]
    lo, hi = np.nanpercentile(sub['logfc'].dropna(), [1, 99])
    _actual[g] = sub.assign(l=sub['logfc'].clip(lo, hi)).groupby('compound')['l'].mean()
actual_logfc = pd.DataFrame(_actual)   # wide: one col per target (compound sets differ -> NaN, fine for KDE)

mlp.plot_objective_kde(_g5, [final_pred, actual_logfc],
                       [SERAC_C['paper'], SERAC_C['gold']],
                       ['predicted (550K)', 'actual (training)']);

In [ ]:
## Visualize directionality: per-objective KDE overlay (full library vs Pareto selections).
_objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
mlp.plot_objective_kde(
    _objs,
    [final_pred, prio_all, prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', 'with TGFB1', 'without TGFB1'],
);

In [ ]:
## select best TGFB1 compounds separately
TGF_sel = final_pred[~final_pred['smiles'].isin(prio_noTGF['smiles'])].sort_values('TGFB1').head(200).reset_index(drop=True)
TGF_sel.shape

In [ ]:
# deliverable = pd.concat([TGF_sel,prio_noTGF]).reset_index(drop=True).drop(['compound','pareto_front'],axis=1)
# deliverable.to_csv(DROPBOX_ML+'predictions/Enamine_Library_15_(Echoes)_working_files/20260624_SingleLow_5Targets_1K_from_550Kenum.csv',sep=',',index=False)

##### Prioritizing 200 at a time

In [ ]:
objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
ctf  = 0.7

topn   = 50
n2take = 200
ANGPTL4_df = pd.concat([final_pred.sort_values(objs[0]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[0]).head(n2take)]).drop_duplicates()
BRD2_df = pd.concat([final_pred.sort_values(objs[1]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[1]).head(n2take)]).drop_duplicates()
EPHB4_df = pd.concat([final_pred.sort_values(objs[2]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[2]).head(n2take)]).drop_duplicates()
PCSK9_df = pd.concat([final_pred.sort_values(objs[3]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[3]).head(n2take)]).drop_duplicates()
TGFB1_df = pd.concat([final_pred.sort_values(objs[4]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[4]).head(n2take)]).drop_duplicates()

deliverable = pd.concat([ANGPTL4_df,BRD2_df,EPHB4_df,PCSK9_df,TGFB1_df]).drop_duplicates().reset_index(drop=True)
deliverable.shape

In [ ]:
mlp.plot_objective_kde(
    objs,
    [final_pred, ANGPTL4_df,prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', objs[0],'without TGFB1'], dpi=100
);

In [ ]:
mlp.plot_objective_kde(
    objs,
    [final_pred, BRD2_df,prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', objs[1],'without TGFB1'], dpi=100
);

In [ ]:
mlp.plot_objective_kde(
    objs,
    [final_pred, EPHB4_df,prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', objs[2],'without TGFB1'], dpi=100
);

In [ ]:
mlp.plot_objective_kde(
    objs,
    [final_pred, PCSK9_df,prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', objs[3],'without TGFB1'], dpi=100
);

In [ ]:
mlp.plot_objective_kde(
    objs,
    [final_pred, TGFB1_df,prio_noTGF],
    [SERAC_C['paper'], SERAC_C['azure'], SERAC_C['ember']],
    ['lib', objs[4],'without TGFB1'], dpi=100
);

In [ ]:
deliverable.to_csv(DROPBOX_ML+'predictions/Enamine_Library_15_(Echoes)_working_files/20260624_SingleLow_5Targets_1K_from_550Kenum.csv',sep=',',index=False)

##### Visualization prioritization for JCS 09.2026

In [ ]:
## params:
# out
enum     = 'data/enumeration/20260608_5Targets_SingleLow/Serac scaffolds broad enumeration_filtered.csv' # input
outMFsmi = 'output/enumeration/20260608_5Targets_SingleLow/MF_features_H236_full.parquet' # MF_features
out550k  = 'output/enumeration/20260608_5Targets_SingleLow/pred_logfc_select_genes.parquet' # the predicted logfc per target
out_sl   = 'output/enumeration/20260608_5Targets_SingleLow/pred_single_low.parquet'
# ml models
SELECT_GENES_DIR = 'output/ML/trained_models/20260617_select_genes' # the ML trained model per gene
SINGLELOW_MODEL  = 'output/ML/trained_models/20260623_SingleLow_Class/SingleLow_RF.joblib' # the ML model to predict single/low

In [ ]:
## get experimental Px data:
MS     = pd.read_parquet(MS_PATH)
df_raw = pd.read_parquet(DFRAW_PATH)

In [ ]:
pred550k_df = pd.read_parquet(out550k)
pred_singlelow_df = pd.read_parquet(out_sl)

final_pred = pd.merge(pred550k_df,pred_singlelow_df[['compound','smiles', 'p_single_low']])
print(final_pred.shape)
final_pred.head(1)

In [ ]:
objs = ['ANGPTL4', 'BRD2', 'EPHB4', 'PCSK9', 'TGFB1', 'p_single_low']
ctf  = 0.7

topn   = 50
n2take = 200
ANGPTL4_df = pd.concat([final_pred.sort_values(objs[0]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[0]).head(n2take)]).drop_duplicates()
BRD2_df = pd.concat([final_pred.sort_values(objs[1]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[1]).head(n2take)]).drop_duplicates()
EPHB4_df = pd.concat([final_pred.sort_values(objs[2]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[2]).head(n2take)]).drop_duplicates()
PCSK9_df = pd.concat([final_pred.sort_values(objs[3]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[3]).head(n2take)]).drop_duplicates()
TGFB1_df = pd.concat([final_pred.sort_values(objs[4]).head(topn),
                        final_pred[final_pred['p_single_low']>ctf].sort_values(objs[4]).head(n2take)]).drop_duplicates()

deliverable = pd.concat([ANGPTL4_df,BRD2_df,EPHB4_df,PCSK9_df,TGFB1_df]).drop_duplicates().reset_index(drop=True)
deliverable.shape

In [ ]:
## predicted vs library-background logfc, per gene
## for each gene: the top-N predicted compounds (most negative predicted logfc) against the
## measured logfc of the whole screened library for that same gene -> is the model picking
## compounds beyond the background distribution?
gene_dfs = {'ANGPTL4': ANGPTL4_df, 'BRD2': BRD2_df, 'EPHB4': EPHB4_df,
            'PCSK9': PCSK9_df, 'TGFB1': TGFB1_df}
PRED_LBL = f'predicted (top {250})'

rows = []
for g, gdf in gene_dfs.items():
    rows.append(pd.DataFrame({'gene': g, 'source': PRED_LBL, 'logfc': gdf.nsmallest(250, g)[g].values}))
    rows.append(pd.DataFrame({'gene': g, 'source': 'library', 'logfc': df_raw.loc[df_raw['genes'] == g, 'logfc'].values}))
viol_df = pd.concat(rows, ignore_index=True)

stats_tools.plot_grouped_boxplot(
    viol_df, group_col='gene', subgroup_col='source', value_col='logfc',
    groups=list(gene_dfs), subgroups=[PRED_LBL, 'library'],
    colors=[SERAC_C['olive'], SERAC_C['paper']],
    show_stars=True, stars_test='ks', ylabel='logfc', width=7, height=5, dpi=150)

In [ ]:
## read the deliverable back from Dropbox (streamed over ssh; no local copy)
deliverable_f = DROPBOX_ML + 'predictions/Enamine_Library_15_(Echoes)_working_files/20260624_SingleLow_5Targets_1K_from_550Kenum.csv'
deliverable = pd.read_csv(_dbx(deliverable_f))
deliverable.shape

#### 20260624 Interface prioritization of compounds to be sent for FBX validation

In [ ]:
dropbox_f = DROPBOX_ML+'interfaces/selection/20260624/20260624_MS_validation_cmps_NN.csv'
full_sel = pd.read_csv(dropbox_f)
full_sel.shape

In [ ]:
stock = pd.read_excel('/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/8_Data_Warehouse/Inventory/Inventory update for SRCBS_20260618.xlsx', sheet_name='Solid(II)')
# non-numeric stock amounts -> 0.0
stock['Stock Amount(mg)'] = pd.to_numeric(stock['Stock Amount(mg)'], errors='coerce').fillna(0.0)

In [ ]:
full_sel[full_sel[ 'Batch Molecule-Batch ID'].isin(stock[stock['Stock Amount(mg)']>0.0]['Batch Molecule Batch ID'])].reset_index(drop=True).shape

In [ ]:
## check the NN selection done by Florent
_ax = sns.boxplot(x='target', y='NN_sim', data=full_sel)
_tmp = _ax.set_xticklabels(_ax.get_xticklabels(), rotation=45, ha='right')

In [ ]:
## split in two plates: 
seed_df = full_sel[full_sel['is_seed']==True].drop(['Molecule Name','is_seed','target','NN_sim','NN_partner','mol_smi'],axis=1).reset_index(drop=True)
analogs_df = full_sel[full_sel['is_seed']==False].drop(['Molecule Name','is_seed','genes','NN_sim','NN_partner','mol_smi','Activity'],axis=1).reset_index(drop=True)

In [ ]:
seed_df.to_csv(DROPBOX_ML+'interfaces/selection/20260624/20260624_MS_validation_FBX_dependency.csv',index=False,sep=',')
analogs_df.to_csv(DROPBOX_ML+'interfaces/selection//20260624/20260624_MS_validation_NN_analogs.csv',index=False,sep=',')

#### 20260709 Single/Low Prioritization of our library

In [ ]:
registry.list()

##### > 20260716 - Predicting 17K library

In [ ]:
## import enamine data:
enum_dbx = _dbx('/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/2_Chemistry/Wuxi/Libraries (design files)/66_SRB-386_amides/66_SRB-386_amides.sdf')
enum = rdkit_tools.get_smiles_df_from_sdf(fn.convert2sdf(enum_dbx), origin='SRB-386_amides',extra_props=['Reactant-ID 1'])

In [ ]:
## predict activity low proba based on MLTrail model
## Score the enumeration (compound + smiles) with the saved low classifier:
##   proba_low = P(ndown in 1-12) -> Px_activity_1_12
## MLTrail re-derives H236 from the SMILES itself, so we only feed compound + smiles.
low_id = registry.list().set_index('experiment_name').loc['Px_activity_1_12_rf', 'id']

# registry.predict takes an in-memory DataFrame directly (no file I/O needed)
enum_pred = enum[['compound', 'smiles']].reset_index(drop=True).copy()
# remove smiles present in training set (compare on CANONICAL smiles to catch equivalent structures)
train_smi = set(registry.load_training_set(low_id)['smiles'].map(rdkit_tools.convert_smiles_to_canonical))
_before   = len(enum_pred)
enum_pred = enum_pred[~enum_pred['smiles'].map(rdkit_tools.convert_smiles_to_canonical).isin(train_smi)].reset_index(drop=True)
print(f'> dropped {_before - len(enum_pred):,} enum compounds whose SMILES are in the training set '
      f'({len(enum_pred):,} remain)')

pred = registry.predict(low_id, enum_pred, smiles_column='smiles', compound_id='compound')
enum_pred['proba_low'] = pred['probability'].values   # probability of the positive class

print(f'> scored {len(enum_pred):,} enum compounds through the low (1-12) MLTrail model')
print(enum_pred['proba_low'].describe().round(3).to_string())

# remove duplicated compounds
enum_pred = pd.merge(enum_pred, enum.drop('origin', axis=1)).drop_duplicates()
enum_pred.head()

In [ ]:
# ## push the scored enum to the company Dropbox (streamed over ssh; WSL path, no local copy)
# dpx_dump = os.path.join(DROPBOX_ML, 'predictions/Discreet_66_SRB-386_amides/20260716_low_1_12_17K.csv')
# fn.push_to_dropbox(enum_pred.drop('proba_nothigh',axis=1), dpx_dump)
# print(f'> pushed {len(enum_pred):,} rows -> {dpx_dump}')

In [ ]:
final_df = enum_pred[ (enum_pred['proba_low']>=0.75) ].sort_values('proba_low').reset_index(drop=True)
final_df.shape

In [ ]:
from numba import njit
@njit(fastmath=True, cache=True)
def tanimoto_dist_numba(a, b):
    dot = np.dot(a, b)
    return 1.0 - dot / (a.sum() + b.sum() - dot + 1e-12)

## binary Morgan fingerprints for the enum (Tanimoto needs binary bits), aligned to proba_low
feat = enum_pred[['compound', 'smiles', 'proba_low']].drop_duplicates('compound').merge(
    rdkit_tools.get_MF_bits_from_df(enum_pred, nBits=2048), on='compound')
X = feat.drop(columns=['compound', 'smiles', 'proba_low']).values.astype(np.float32)

emb = oTSNE(metric=tanimoto_dist_numba, neighbors='pynndescent', n_jobs=-1,
            n_components=2, perplexity=30, random_state=42).fit(X)

In [ ]:
e_df = feat[['compound', 'smiles', 'proba_low']].copy()
e_df['t1'], e_df['t2'] = emb[:, 0], emb[:, 1]

# selected = predicted low-active above threshold
proba_ctf = 0.75
e_df_h = e_df[e_df['proba_low'] >  proba_ctf].copy()   # selected
e_df_l = e_df[e_df['proba_low'] <= proba_ctf].copy()   # rest
print(f'> selected {len(e_df_h):,} / {len(e_df):,} enum compounds at proba_low > {proba_ctf}')

## plotting — silent -> active (low -> high), matching the existing palette
active_c, silent_c = SERAC_C['azure'], SERAC_C['paper']
cmap_ra = LinearSegmentedColormap.from_list('silent_to_active', [silent_c, '#ffffff', active_c])

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
ax.scatter(e_df_l['t1'], e_df_l['t2'], c=e_df_l['proba_low'], cmap=cmap_ra, vmin=0, vmax=1,
           s=20, alpha=1, edgecolors='black', linewidths=0.0)                        # rest
sc = ax.scatter(e_df_h['t1'], e_df_h['t2'], c=e_df_h['proba_low'], cmap=cmap_ra, vmin=0, vmax=1,
                s=20, alpha=1, edgecolors='black', linewidths=0.5)                   # selected (outlined)
plt.colorbar(sc, ax=ax, label='predicted p(low activity)')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.title(f'Enum chemical space (Tanimoto t-SNE) — {len(e_df_h):,} selected at proba_low > {proba_ctf}')
plt.tight_layout()

In [ ]:
## compare enum's chemical space distance to training:
## load the low-model's TRAINING set from MLTrail, then use Rdkit_tools' NN helpers to get each enum
## compound's nearest-neighbour Tanimoto distance to training — same applicability-domain view as the
## ActivityClass cell, now for the 17K virtual library.
low_id    = registry.list().set_index('experiment_name').loc['Px_activity_1_12', 'id']
train_set = registry.load_training_set(low_id).rename(columns={'compound_id': 'compound'})  # compound, smiles, label
print(f'> loaded {len(train_set):,} training compounds from MLTrail (model id {low_id})')

train_MF = rdkit_tools.get_MF_bits_from_df(train_set)
enum_MF  = rdkit_tools.get_MF_bits_from_df(enum_pred)

# per-enum nearest-neighbour distance to training (chunked over training -> memory-safe for 17K)
enum_nn   = rdkit_tools.compute_NNs_in_chunks(enum_MF, train_MF, chunk_size=1000, top=1)
enum_dist = (enum_nn.rename(columns={'distance': 'nn_dist'})[['compound', 'nn_dist']]
             .merge(enum_pred[['compound', 'proba_low']], on='compound'))

# training's own NN spread (exclude self): Rdkit_tools distance matrix, mask the diagonal
dmat = rdkit_tools.get_RDKiTTanimoto_distance_matrix(train_MF, train_MF).to_numpy().copy()  # writable
np.fill_diagonal(dmat, np.inf)
train_self = dmat.min(axis=1)
edge = np.quantile(train_self, 0.95)                                  # 'edge of domain' reference
frac_out = (enum_dist['nn_dist'] > edge).mean()

In [ ]:
print(f"> enum NN dist: median={enum_dist.nn_dist.median():.2f}  "
      f"[{enum_dist.nn_dist.min():.2f}, {enum_dist.nn_dist.max():.2f}]  |  "
      f"training 95th-pct NN dist (domain edge)={edge:.2f}")
print(f"> {frac_out:.1%} of enum sits BEYOND the training's 95th-pct NN distance (out-of-domain)")

## plot: (L) distance distributions, (R) enum proba_low vs distance to training
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
a1.hist(train_self, bins=40, density=True, alpha=0.6, color=SERAC_C['azure'], label='training <-> training')
a1.hist(enum_dist['nn_dist'], bins=40, density=True, alpha=0.6, color=SERAC_C['ember'], label='enum -> training')
a1.axvline(edge, color='k', ls='--', lw=1, label=f'domain edge (0.95q = {edge:.2f})')
a1.set_xlabel('nearest-neighbour Tanimoto distance to training'); a1.set_ylabel('density')
a1.set_title('Chemical-space distance: enum vs training'); a1.legend()

a2.scatter(enum_dist['nn_dist'], enum_dist['proba_low'], s=6, alpha=0.2, color=SERAC_C['ember'])
a2.axvline(edge, color='k', ls='--', lw=1, label=f'domain edge ({edge:.2f})')
a2.axhline(0.5, color='grey', ls='--', lw=1)
a2.set_xlabel('nearest-neighbour Tanimoto distance to training'); a2.set_ylabel('predicted p(ndown in [1,12])')
a2.set_title('Enum: model probability vs distance to training'); a2.legend()
plt.tight_layout()

In [ ]:
## visually check top predictions: and show nearest neighbor:
## top-N enum compounds by proba_low, each paired with its nearest training neighbour (from enum_nn).
## grid rows alternate ENUM (p_low) | its NN (Tanimoto distance) so you can eyeball the analogy.
TOPN = 3
top = (enum_pred.merge(enum_nn[['compound', 'NN', 'distance']], on='compound')
       .sort_values('proba_low', ascending=False).head(TOPN))
train_smi = train_set.drop_duplicates('compound').set_index('compound')['smiles']

# interleave [enum, its NN] so mols_per_row=2 puts each pair on one row
rows = []
for _, r in top.iterrows():
    rows.append({'label': f"ENUM {r['compound']}  p={r['proba_low']:.2f}", 'smiles': r['smiles']})
    rows.append({'label': f"NN {r['NN']}  Td={r['distance']:.2f}",         'smiles': train_smi.get(r['NN'])})
pairs = pd.DataFrame(rows)

rdkit_tools.draw_compounds_grid(pairs, smiles_col='smiles', compound_col='label',
                                mols_per_row=2, sub_img_size=(320, 260))

##### > 20260716 - Predicting Florent's 2K compounds

In [ ]:
## predict Florent's 2K compounds:
## import enamine data:
flor_dbx = _dbx(DROPBOX_ML+ '/predictions/Discreet_66_SRB-386_amides/20260709_MultiClassActivity_pred_from17K_awayFromSilent_2000.sdf')
flor = rdkit_tools.get_smiles_df_from_sdf(fn.convert2sdf(flor_dbx), origin='SRB-386_amides',extra_props=['Reactant-ID 1'])

In [ ]:
## predict activity low proba based on MLTrail model
## Score the enumeration (compound + smiles) with the saved low classifier:
##   proba_low = P(ndown in 1-12) -> Px_activity_1_12
## MLTrail re-derives H236 from the SMILES itself, so we only feed compound + smiles.
low_id = registry.list().set_index('experiment_name').loc['Px_activity_1_12', 'id']

# registry.predict takes an in-memory DataFrame directly (no file I/O needed)
flor_pred = flor[['compound', 'smiles']].reset_index(drop=True).copy()
pred = registry.predict(low_id, flor_pred, smiles_column='smiles', compound_id='compound')
flor_pred['proba_low'] = pred['probability'].values   # probability of the positive class

print(f'> scored {len(flor_pred):,} enum compounds through the low (1-12) MLTrail model')
print(flor_pred['proba_low'].describe().round(3).to_string())
flor_pred.head()

In [ ]:
flor_pred[flor_pred['proba_low']>0.75]

##### > 20261005 - Echoes compound from 11K virtual library

In [ ]:
REPORT_DIR = params.DROPBOX_ML + params.STANDOUT_REPORT_DBX
CSV_NAME   = f'20261005_Echoes_compounds.csv'
CSV_COLS   = ['target', 'compound', 'plate', 'logfc', 'pvalue', 'layer', 'd_any', 'd_bulk',
              'density', 'n_frontier', 'ndown', 'activity', 'partners_down', 'partners_logfc',
              'rank', 'drawn']   # drawn = its volcano is plotted (the top STANDOUT_REPORT_MAX)
# TARGETS    = hot_tbl['symbol'].dropna().unique()
Path(params.STANDOUT_REPORT_LOC).mkdir(parents=True, exist_ok=True)
rows, echoes, skipped = [], [], {}

In [ ]:
## Potentially under-appreciated echoes compounds:
echoes = pd.read_csv(_dbx(REPORT_DIR + CSV_NAME))

## enumerated lib:
enum_echo_dbx = _dbx(DROPBOX_ML+ '/virtual libraries/enum_carbamates_10996.sdf')
echo_enum = rdkit_tools.get_smiles_df_from_sdf(fn.convert2sdf(enum_echo_dbx), origin='carbamates',extra_props=['Reactant-ID 1'])
echo_enum.head(1)

In [ ]:
echoes_lib = data.serac_df[data.serac_df['compound'].isin(echoes['compound'])][['compound','smiles']]
echoes_lib.head(1)

In [ ]:
## extract nearest neighburs to echoes compounds:
TOP_NN    = 10      # nearest neighbours kept per echo compound — raise it for a shortlist each
TDIST_CTF = 0.4

# make smiles canonical  
echoes_lib['smiles'] = echoes_lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))
echo_enum['smiles']  = echo_enum['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))

# clean dataset and drop duplicates:
echoes_lib = echoes_lib.drop_duplicates('compound').reset_index(drop=True)
echo_enum = echo_enum.drop_duplicates('smiles').reset_index(drop=True)

# get MFs and compute distance matrix and compute NNs:
lib_MF, enum_MF = rdkit_tools.get_MF_bits_from_df(echoes_lib), rdkit_tools.get_MF_bits_from_df(echo_enum)
d = rdkit_tools.get_RDKiTTanimoto_distance_matrix(lib_MF, enum_MF)
echo_nn = rdkit_tools.get_NN_from_dist_matrix(d, top=TOP_NN).sort_values(['compound', 'distance'])

# get targets associated with echo lib compunds
echo_nn = (echo_nn.merge(echoes.groupby('compound')['target'].agg('; '.join), on='compound', how='left')
           .sort_values(['target','distance']).reset_index(drop=True))

# remove identical compounds
echo_nn = echo_nn[echo_nn['distance']>0].reset_index(drop=True)

## select subset based on tanimoto distance
sel = echo_nn[echo_nn['distance'] < TDIST_CTF]

In [ ]:
## one html report: per gene, each echo volcano with its neighbour grid underneath
NN_NAME = f'{date.today():%Y%m%d}_HT_NN_{TDIST_CTF}.html'
NN_LOC  = os.path.join(params.STANDOUT_REPORT_LOC, NN_NAME)

# lookups: the volcano of a (compound, plate) pair, and the smiles of a neighbour
uc  = (data.df_raw.assign(plate=data.df_raw['MSPlate'].fillna(''))
       .drop_duplicates(['compound', 'plate']).set_index(['compound', 'plate'])['uniquecontrast'])
smi = echo_enum.drop_duplicates('compound').set_index('compound')['smiles']

# one row per gene and compound: split the shared targets, then take back plate and activity
nn_rows = (sel.assign(gene=sel['target'].str.split('; ')).explode('gene')
           .merge(echoes.rename(columns={'target': 'gene'})[['gene', 'compound', 'plate', 'ndown',
                                                             'activity', 'd_any']],
                  on=['gene', 'compound'], how='left')
           .assign(plate=lambda x: x['plate'].fillna('')).sort_values(['gene', 'distance']))

# page header and the gene index
genes  = sorted(nn_rows['gene'].unique())
blocks = [f'<h1>Hot-target echoes — virtual neighbours within Tanimoto {TDIST_CTF}</h1>',
          f'<p>{nn_rows["compound"].nunique()} echo compounds over {len(genes)} genes, '
          f'{nn_rows["NN"].nunique()} distinct virtual neighbours. {date.today()}.</p>',
          '<p>' + ' · '.join(f'<a href="#{x}">{x}</a>' for x in genes) + '</p>']

# one section per gene, one volcano plus neighbour grid per echo compound
n = 0
for gene in tqdm(genes, desc='genes'):
    g = nn_rows[nn_rows['gene'] == gene]
    shell = fn.string_first_shell(gene, species=params.STRING_SPECIES, min_score=params.STRING_MIN_SCORE,
                                  limit=params.STRING_MAX_PARTNERS, cache_dir=params.STRING_CACHE)
    blocks.append(f'<hr style="margin:48px 0 24px"><h1 id="{gene}">{gene}</h1>'
                  f'<p>{g["compound"].nunique()} echo compounds · {len(g)} neighbours · gold rings '
                  f'= the {len(shell)} STRING first-shell partners.</p>')
    for cpd, c in g.groupby('compound', sort=False):
        r   = c.iloc[0]
        fig = fn.plot_volcano_pareto(data.df_raw, uc.loc[(cpd, r['plate'])], highlight=gene,
                                     max_layer=params.STANDOUT_MAX_LAYER, up_color=params.ACTIVE_C,
                                     down_color=params.SILENT_C, partners=shell['partner'].tolist(),
                                     compound=cpd, struct_dir=params.SRB_PNG_DIR, struct_size=0.45)
        blocks.append(f"<h2>{cpd} — {r['plate']} — {r['activity']} (ndown {r['ndown']:.0f}) "
                      f"— d_any {r['d_any']:.3f}</h2>")
        blocks.append(fig.to_html(full_html=False, include_plotlyjs='cdn' if not n else False))
        blocks.append(f'<p><b>{len(c)} nearest neighbour(s)</b> within Tanimoto {TDIST_CTF}:</p>'
                      + fn.mols_grid_svg(c['NN'].map(smi),
                                         [f'{x}  d={y:.3f}' for x, y in zip(c['NN'], c['distance'])]))
        n += 1

# write and push
Path(NN_LOC).write_text('<!doctype html><meta charset="utf-8">'
                        f'<title>Hot-target echo neighbours (d < {TDIST_CTF})</title>'
                        '<body style="font-family:system-ui;max-width:1100px;margin:24px auto">'
                        + '\n'.join(blocks) + '</body>', encoding='utf-8')
# fn.push_rclone(NN_LOC, REPORT_DIR + NN_NAME)

nn_rows.groupby('gene').agg(compounds=('compound', 'nunique'), neighbours=('NN', 'size'),
                            best_d=('distance', 'min')).round(3)

In [ ]:
## output a final sdf of NN including the ['Reactant-ID 1']
NN_SDF = NN_LOC.replace('.html', '.sdf')

# one record per unique neighbour, with the echo compounds and targets it was matched to
hits = nn_rows.groupby('NN').agg(echo_compound=('compound', lambda s: '; '.join(sorted(set(s)))),
                                 target=('gene', lambda s: '; '.join(sorted(set(s)))),
                                 distance=('distance', 'min'), n_echoes=('compound', 'nunique'))
sdf_df = (echo_enum.drop_duplicates('compound').set_index('compound')
          .loc[hits.index, ['smiles', 'Reactant-ID 1', 'origin']]
          .join(hits).reset_index().rename(columns={'NN': 'compound'}))

# write and push: every non-smiles column becomes an sdf tag
n_sdf = rdkit_tools.write_filtered_enum_sdf(sdf_df, out_path=NN_SDF)
# fn.push_rclone(NN_SDF, REPORT_DIR + os.path.basename(NN_SDF))

# roundtrip: read the sdf back and recover the echo_enum rows
nn_back  = rdkit_tools.get_smiles_df_from_sdf(NN_SDF, origin='carbamates',
                                              extra_props=['Reactant-ID 1', 'echo_compound',
                                                           'target', 'distance'])
nn_check = echo_enum[echo_enum['compound'].isin(nn_back['compound'])].reset_index(drop=True)
# every written record comes back to exactly one echo_enum row
assert len(nn_back) == len(nn_check) == n_sdf

nn_check

##### > 20261009 - Diverse Glue selection for ePSA and ChromLogD 

In [ ]:
## generate 2D embedding of the library
PERPLEXITY = 30
N_JOBS     = 8
SEED       = 0

# clean library: one row per compound
lib = data.serac_df[['compound', 'smiles']].drop_duplicates('compound').reset_index(drop=True)

# remove devalidated and contaminants compounds: cm2rm = the CM2RM_PARTS lists
lib = lib[~lib['compound'].isin(data.cm2rm)].reset_index(drop=True)

# canonical smiles
lib['smiles'] = lib['smiles'].progress_apply(lambda x: rdkit_tools.convert_smiles_to_canonical(x))

# morgan fingerprints
lib_MF = rdkit_tools.get_MF_bits_from_df(lib, v=True)

# 3d tsne on tanimoto: jaccard on binary bits is the same metric
E = oTSNE(n_components=2, metric='jaccard', neighbors='pynndescent', perplexity=PERPLEXITY,
          n_jobs=N_JOBS, random_state=SEED).fit(lib_MF.drop(columns='compound').to_numpy(np.float32))

lib_2d = lib_MF[['compound']].assign(e1=E[:, 0], e2=E[:, 1])
print(lib_2d.shape)
lib_2d.head(3)

In [ ]:
## add physchem properties
FEATS = ['e1', 'e2', 'Hba', 'Hbd', 'MW', 'TPSA', 'LogP', 'NRB']

# rdkit physchem per compound
lib_2d = (lib_2d[['compound', 'e1', 'e2']]
          .merge(rdkit_tools.compute_properties_from_smiles(lib), on='compound', how='left'))

# drop the compounds with no structure
lib_2d = lib_2d.dropna(subset=FEATS).reset_index(drop=True)

# scale every kmeans feature to 0-1
lib_2d[FEATS] = MinMaxScaler().fit_transform(lib_2d[FEATS])

lib_2d.head(3)

In [ ]:
## select 50 representative structures via k-means
N_REP = 150

# cluster on every scaled attribute
km = KMeans(n_clusters=N_REP, random_state=SEED, n_init=10).fit(lib_2d[FEATS])
lib_2d['cluster'] = km.labels_
lib_2d['dist'] = np.linalg.norm(lib_2d[FEATS].to_numpy() - km.cluster_centers_[km.labels_], axis=1)

# medoid = the real compound closest to its cluster centroid
lib_2d['medoid'] = lib_2d.index.isin(lib_2d.groupby('cluster')['dist'].idxmin()).astype(int)

lib_2d[lib_2d['medoid']==1].head(3)

In [ ]:
## show 2D tsne of selected compounds:
# one colour per cluster, no legend for 50 of them
palette = plt.cm.hsv(np.linspace(0, 1, N_REP))
ax = stats_tools.plot_multi_class_to_embedding(lib_2d, label='cluster', e1='e1', e2='e2', s=6, leg=False,
                                               colors=palette, xlabel='TSNE-1', ylabel='TSNE-2')

# the medoid of each cluster as a star, filled with its own cluster colour
med = lib_2d[lib_2d['medoid'] == 1]
colour_of = dict(zip(lib_2d['cluster'].unique(), palette))   # the order the helper assigns colours in
ax.scatter(med['e1'], med['e2'], marker='*', s=250, zorder=3,
           c=[colour_of[k] for k in med['cluster']], edgecolors='black', linewidths=2)
ax.set_title(f'{lib_2d["cluster"].nunique()} clusters, {len(med)} representative structures')

In [ ]:
## Extract the 50 representative compounds as .sdf:
SEL_DBX = params.DROPBOX_ML + f'predictions/20261009_ePSA_LogD_selection/'
SEL_SDF = f'output/MS/{date.today():%Y%m%d}_{N_REP}_representative_glues.sdf'

# the medoids with their real, unscaled properties
reps = lib_2d.loc[lib_2d['medoid'] == 1, ['compound', 'cluster']].merge(lib, on='compound')
reps = reps.merge(rdkit_tools.compute_properties_from_smiles(reps), on='compound')

# write and push: every non-smiles column becomes an sdf tag
rdkit_tools.write_filtered_enum_sdf(reps, out_path=SEL_SDF)
fn.push_rclone(SEL_SDF, SEL_DBX + os.path.basename(SEL_SDF))

print(reps.shape)
reps.head(3)

##### > 20261103 - Glue selection for Target-centric assay:

In [ ]:
## reopen the vault and load the high model
registry = Registry.from_default()
high_id  = int(registry.list().set_index('experiment_name').loc[params.HIGH_MODEL_NAME, 'id'])
bundle   = joblib.load(registry.details(high_id)['model_path']) #_> dict={model,feature_cols,positive,...}

In [30]:
## add the inventory actual amount to serac_df:
cmp_inventory  = DROPBOX_LOCAL_ROOT + 'Serac_team/8_Data_Warehouse/Inventory/Inventory update for SRCBS_20261101_solid_extract.xlsx'
PROBA_HIGH_MAX = 0.1    # compounds without activity above this proba_high are removed
MIN_MG         = 1.0
N_REP          = 8000     # compounds selected in total: KEEP_LIBS ones + k-means picks of the rest
PERPLEXITY     = 30
N_JOBS         = 8
SEED           = 0
KEEP_LIBS      = ['Tango']    # libraries kept whole; they count toward N_REP

# actual amount per compound, summed over its batches
inv = pd.read_excel(_dbx(cmp_inventory)).groupby('Molecule Name')['Actual amount'].sum() #_> series=index:Molecule Name, values:Actual amount

# map onto serac_df by compound
serac_df['Actual amount'] = serac_df['compound'].map(inv)

# add Px data:
final = pd.merge(serac_df,MS[['compound','date','ndown','activity']],how='left',on='compound').drop_duplicates(['compound'])
print(f'> total AK and AJ {final.shape}')

# keep compounds with sufficient amounts in inventory
final = final[(final['Actual amount'] > MIN_MG)]
print(f'> with Solid amount > 0 {final.shape}')

# Removing high activity compounds:
final = final[(final['activity']!='High (>25)')]
print(f'> without Px High > 0 {final.shape}')

if 11 < 3:
    # P(high) from the cached MF_features, for the compounds with no activity
    X = final.loc[final['activity'].isna(), ['compound']].merge(MF_features, on='compound')
    p_high = pd.Series(bundle['model'].predict_proba(X[bundle['feature_cols']])[:, 1], index=X['compound'])
    final = final.assign(proba_high=final['compound'].map(p_high))

    # remove compound without activity labels that have high probability of being High
    final = final[~(final['proba_high'] > PROBA_HIGH_MAX)]
    print(f'> without proba_high > {PROBA_HIGH_MAX} {final.shape}')


# extract representative compounds:
# tsne of the cached morgan bits: jaccard on binary bits is tanimoto
fin_MF = final[['compound']].merge(MF.drop_duplicates('compound'), on='compound')
E = oTSNE(n_components=2, metric='jaccard', neighbors='pynndescent', perplexity=PERPLEXITY,
          n_jobs=N_JOBS, random_state=SEED).fit(fin_MF.drop(columns='compound').to_numpy(np.float32))
final = final.assign(e1=E[:, 0], e2=E[:, 1])

# KEEP_LIBS compounds go in whole; k-means on the rest picks N_REP minus them
keep = final['Lib Flavor'].isin(KEEP_LIBS).to_numpy()
km = KMeans(n_clusters=N_REP - keep.sum(), random_state=SEED, n_init=10).fit(E[~keep])

# medoid = the real compound closest to its cluster centroid
final.loc[~keep, 'cluster'] = km.labels_
final.loc[~keep, 'dist'] = np.linalg.norm(E[~keep] - km.cluster_centers_[km.labels_], axis=1)
final['medoid'] = final.index.isin(final.groupby('cluster')['dist'].idxmin()).astype(int)
print(f'> representative compounds {final[final["medoid"] == 1].shape}')

# selected = every KEEP_LIBS compound left in final + the representatives of the rest
final['selected'] = ((final['medoid'] == 1) | keep).astype(int)
final = final[ final['selected'] == 1 ]

print(f'> selected compounds after k-means {final.shape}')

> total AK and AJ (14086, 17)
> with Solid amount > 0 (12575, 17)
> without Px High > 0 (11636, 17)
> representative compounds (7161, 22)
> selected compounds after k-means (8000, 23)
